# <center> Homework 137

In [1]:
import tensorflow as tf
import tf_model
import tf_data
from importlib import reload
import tensorflow_datasets as tfds
from pathlib import Path
import numpy as np

2026-02-26 21:59:23.631445: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-26 21:59:23.773609: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-26 21:59:31.785598: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/home/zdravko/Machine_Learning_Intern/venv/lib/python3.13/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):
2026-02-26 21:59:39.092719: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit

## Task 1
да се тестват примера в книгата за Bidirectional RNN

In [5]:
url = "https://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip"
path = tf.keras.utils.get_file("spa-eng.zip", origin=url, cache_dir="datasets/spa-eng", extract=True)

2638744/2638744 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [2]:
# text = (Path('/tmp/.keras/datasets/spa-eng_extracted/spa-eng').with_name("spa-eng") / "spa.txt").read_text()
text = (Path('datasets/spa-eng/datasets/spa-eng_extracted/spa-eng').with_name("spa-eng") / "spa.txt").read_text()

In [3]:
text = text.replace("¡", "").replace("¿", "")
pairs = [line.split("\t") for line in text.splitlines()]
np.random.shuffle(pairs)
sentences_en, sentences_es = zip(*pairs)

In [4]:
vocab_size = 1000
max_length = 50

text_vec_layer_en = tf.keras.layers.TextVectorization(
                        vocab_size, output_sequence_length=max_length)

text_vec_layer_es = tf.keras.layers.TextVectorization(
                        vocab_size, output_sequence_length=max_length)

text_vec_layer_en.adapt(sentences_en)
text_vec_layer_es.adapt([f"startofseq {s} endofseq" for s in sentences_es])

2026-02-26 21:59:48.219230: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 23080128 exceeds 10% of free system memory.


In [6]:
X_train = tf.constant(sentences_en[:100_000])
X_valid = tf.constant(sentences_en[100_000:])

X_train_dec = tf.constant([f"startofseq {s}" for s in sentences_es[:100_000]])
X_valid_dec = tf.constant([f"startofseq {s}" for s in sentences_es[100_000:]])

Y_train = text_vec_layer_es([f"{s} endofseq" for s in sentences_es[:100_000]])
Y_valid = text_vec_layer_es([f"{s} endofseq" for s in sentences_es[100_000:]])

2026-02-26 21:57:00.981206: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 40000000 exceeds 10% of free system memory.


In [15]:
embed_size = 128

# INPUTS
encoder_inputs = tf.keras.layers.Input(shape=[], dtype=tf.string)
decoder_inputs = tf.keras.layers.Input(shape=[], dtype=tf.string)

# VECTORIZATION
encoder_input_ids = text_vec_layer_en(encoder_inputs)
decoder_input_ids = text_vec_layer_es(decoder_inputs)

# EMBEDDING
encoder_embedding_layer = tf.keras.layers.Embedding(vocab_size, embed_size, mask_zero=True)
decoder_embedding_layer = tf.keras.layers.Embedding(vocab_size, embed_size, mask_zero=True)

encoder_embeddings = encoder_embedding_layer(encoder_input_ids)
decoder_embeddings = decoder_embedding_layer(decoder_input_ids)

# ENCODER
encoder = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(256, return_state=True))
encoder_outputs, *encoder_state = encoder(encoder_embeddings)

encoder_state = [tf.keras.layers.concatenate(encoder_state[::2], axis=-1),  # short-term (0 & 2)
                 tf.keras.layers.concatenate(encoder_state[1::2], axis=-1)] # long-term (1 & 3)

# DECODER
decoder = tf.keras.layers.LSTM(512, return_sequences=True)
decoder_outputs = decoder(decoder_embeddings, initial_state=encoder_state)

# OUTPUT
output_layer = tf.keras.layers.Dense(vocab_size, activation="softmax")
Y_proba = output_layer(decoder_outputs)

nmt_model = tf.keras.Model(inputs=[encoder_inputs, decoder_inputs], outputs=[Y_proba])

In [12]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [16]:
nmt_model.compile(loss="sparse_categorical_crossentropy", optimizer="nadam", metrics=["accuracy"])

callbacks = [tf.keras.callbacks.ModelCheckpoint('/content/drive/MyDrive/ml_models/nmt_bidirectional_model.keras')]
nmt_model.fit((X_train, X_train_dec), Y_train,
              epochs=10,
              validation_data=((X_valid, X_valid_dec), Y_valid),
              callbacks=callbacks)

Epoch 1/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 85s 26ms/step - accuracy: 0.0536 - loss: 3.4103 - val_accuracy: 0.0797 - val_loss: 1.9357
Epoch 2/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 77s 25ms/step - accuracy: 0.0832 - loss: 1.7851 - val_accuracy: 0.0902 - val_loss: 1.5069
Epoch 3/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 78s 25ms/step - accuracy: 0.0940 - loss: 1.3683 - val_accuracy: 0.0940 - val_loss: 1.3669
Epoch 4/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 82s 25ms/step - accuracy: 0.1003 - loss: 1.1471 - val_accuracy: 0.0955 - val_loss: 1.3244
Epoch 5/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 77s 25ms/step - accuracy: 0.1050 - loss: 0.9898 - val_accuracy: 0.0959 - val_loss: 1.3248
Epoch 6/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 78s 25ms/step - accuracy: 0.1090 - loss: 0.8639 - val_accuracy: 0.0957 - val_loss: 1.3470
Epoch 7/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 77s 25ms/step - accuracy: 0.1123 - loss: 0.7617 - val_accuracy: 0.0952 - val_loss: 1.3812
Epoch 8/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 78s 25ms/step - accuracy: 0.1150 -

In [14]:
def translate(sentence_en):
    translation = ""
    for word_idx in range(max_length):
        X = tf.constant([sentence_en])  # encoder input
        X_dec = tf.constant(["startofseq " + translation])  # decoder input
        y_proba = nmt_model.predict((X, X_dec))[0, word_idx]  # last token's probas
        predicted_word_id = np.argmax(y_proba)
        predicted_word = text_vec_layer_es.get_vocabulary()[predicted_word_id]
        if predicted_word == "endofseq":
            break
        translation += " " + predicted_word
    return translation.strip()

In [18]:
translate("I like soccer")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 380ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step


'me gusta el fútbol'

In [19]:
translate("I like soccer and also going to the beach")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step


'me gusta cuando los [UNK] voy a la escuela'

In [20]:
translate("I love watching movie")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step


'me encanta ver televisión'

## Task 2
да се имплементира класа Bidirectional

In [38]:
reload(tf_data)
from tf_data import Dataset


raw_train_set, raw_valid_set, raw_test_set = tfds.load(
    name="imdb_reviews",
    split=["train[:50%]", "train[90%:]", "test"],
    as_supervised=True
)

tf.random.set_seed(42)
adapt_set = raw_train_set.shuffle(5000, seed=42).batch(32).prefetch(1)

train_set = Dataset(raw_train_set).shuffle(5000, seed=42).batch(32).prefetch(1)
valid_set = Dataset(raw_valid_set).batch(32).prefetch(1)
test_set  = Dataset(raw_test_set).batch(32).prefetch(1)

In [42]:
reload(tf_model)
from tf_model import *

vocab_size = 1000
text_vec_layer = TextVectorization(max_tokens=vocab_size)
text_vec_layer.adapt(adapt_set.map(lambda reviews, labels: reviews))

da


In [45]:
min_len = float('inf')

for item in train_set:
    t_dim = text_vec_layer.call(item[0]).shape[1]
    if t_dim < min_len:
        min_len = t_dim

min_len        

2026-02-27 01:08:47.490122: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


356

In [50]:
reload(tf_model)
from tf_model import *

vocab_size = 1000
text_vec_layer = TextVectorization(max_tokens=vocab_size)
text_vec_layer.adapt(adapt_set.map(lambda reviews, labels: reviews))

embed_size = 32
tf.random.set_seed(42)

model = Sequential([
    Input([]),
    text_vec_layer,
    Lambda(lambda X: X[:, :min_len]),
    Embedding(vocab_size, embed_size, mask_zero=True),
    Bidirectional(GRU(32)),
    Dense(1, activation="sigmoid")
])


model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
history = model.fit(train_set, validation_data=valid_set, epochs=2)

da
Epoch 1/2


0it [00:00, ?it/s]

Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name node, self, outside of any statement?
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name node, self, outside of any statement?
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name node, self, outside of any statement?
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name node, activation, outside of any statement?
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


...

Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name node, metric, outside of any statement?
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


391it [10:51,  1.67s/it]
2026-02-27 01:41:45.678766: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


    - loss: 0.694 - accuracy: 0.513
    - val_loss: 0.693 - val_accuracy: 0.517
    - Learning Rate: 0.001
Epoch 2/2


391it [11:16,  1.73s/it]


    - loss: 0.693 - accuracy: 0.516
    - val_loss: 0.693 - val_accuracy: 0.530
    - Learning Rate: 0.001


## Task 3
да се оптимизира функциата translate от учебника така че да ползва beam search

In [5]:
nmt_model = tf.keras.models.load_model('language_models/nmt_bidirectional_model.keras')

In [24]:
translate('soccer is my favorite sport')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 166ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 156ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 128ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 127ms/step


'mi [UNK] es la [UNK] favorita'

In [16]:
def translate_beam_search(sentence_en, k=3, prob_threshold=0.01):
    final_candidats_sen  = []
    final_candidats_prob = []
    X = tf.constant([sentence_en]) 

    vocab = np.array(text_vec_layer_es.get_vocabulary())
    base = {'': 1}

    for word_idx in range(max_length): 
        print(word_idx)
        new_base = {}

        for sentence, sen_prob in base.items():

            X_dec = tf.constant(["startofseq " + sentence])  
            y_proba = nmt_model.predict((X, X_dec))[0, word_idx]  
            
            top_proba_inxs = np.argsort(y_proba)[-k:]
            top_proba = y_proba[top_proba_inxs]
            top_predicted_word = vocab[top_proba_inxs]

            for word, prob in zip(top_predicted_word, top_proba):
                if word == "endofseq":
                    final_candidats_sen .append(sentence)
                    final_candidats_prob.append(sen_prob)
                    continue

                if prob < prob_threshold:
                    continue

                new_base[sentence + " " + word] = sen_prob * prob

        base = new_base
        if not len(base):
            break

    translation = final_candidats_sen[np.argmax(final_candidats_prob)]
    return translation.strip()

In [14]:
translate_beam_search('I like soccer', k=2, prob_threshold=0.01)

0
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 359ms/step
1
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 222ms/step
2
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 199ms/step
3
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 271ms/step
4
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 192ms/step


'me gusta el fútbol'

In [ ]:
translate_beam_search('soccer is my favorite sport', k=2)

0
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 255ms/step
1
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 211ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 179ms/step
2
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 211ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 128ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step
3
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 127ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step
4
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 232ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 163ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 133ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 130ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step
5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step
1/1 ━━━━━━━━━

'el fútbol es mi [UNK] favorita'

In [23]:
translate_beam_search('My book is very funny and I love beach', k=2, prob_threshold=0.1)

0
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 389ms/step
1
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 250ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 206ms/step
2
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 252ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 340ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 212ms/step
3
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 178ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 325ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 214ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 261ms/step
4
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 250ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 247ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 221ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 160ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 177ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 159ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 179ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 382ms/step
5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 158ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 162ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 130ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 127ms/step
1/1 ━━━━━━━━━

'mi libro es muy alto y para mí afuera'

My book is very tall and outside for me

In [36]:
def translate_beam_search_v2(sentence_en, k=3):
    final_candidats_sen  = []
    final_candidats_prob = []
    X = tf.constant([sentence_en]) 

    vocab = np.array(text_vec_layer_es.get_vocabulary())
    base = {'': 1}

    for word_idx in range(max_length): 
        print(word_idx)
        all_cond_probs = []
        all_cond_words = []
        all_sentences  = []

        for sentence, sen_prob in base.items():

            X_dec = tf.constant(["startofseq " + sentence])  
            y_proba = nmt_model.predict((X, X_dec))[0, word_idx]  
            
            cond_proba = sen_prob * y_proba
            top_proba_inxs = np.argsort(cond_proba)[-k:]
            top_proba = cond_proba[top_proba_inxs]
            top_predicted_word = vocab[top_proba_inxs]

            all_cond_probs.extend(top_proba)
            all_cond_words.extend(top_predicted_word)
            all_sentences.extend([sentence] * k)

        top_k_cond_inxs = np.argsort(all_cond_probs)[-k:]
        top_k_cond_probs = np.array(all_cond_probs)[top_k_cond_inxs]
        top_k_words = np.array(all_cond_words)[top_k_cond_inxs]
        top_k_sen = np.array(all_sentences)[top_k_cond_inxs]

        base = {}
        for word, prob, sen in zip(top_k_words, top_k_cond_probs, top_k_sen):
            if word == "endofseq":
                final_candidats_sen .append(sen)
                final_candidats_prob.append(prob)
                continue

            base[sen + " " + word] = prob

        if not len(base):
            break

    translation = final_candidats_sen[np.argmax(final_candidats_prob)]
    return translation.strip()

In [29]:
translate_beam_search_v2('My book is very funny and I love beach', k=2)

0
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 287ms/step
1
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 441ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 182ms/step
2
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 186ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 198ms/step
3
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 200ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 184ms/step
4
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 185ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 183ms/step
5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 248ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 171ms/step
6
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 172ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 172ms/step
7
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 214ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 199ms/step
8
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 270ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 128ms/step
9
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step
10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 178ms/step
11
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 166ms/step
12
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 140ms/step
13
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 133ms/step
14
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 264ms/step
15
1/1 ━━━━━━━━━━━━━━━━━━━━ 0

'mi libro es muy alto y para mí afuera'

In [37]:
translate_beam_search_v2('My car is new', k=2)

0
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 185ms/step
1
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 111ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step
2
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 152ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step
3
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 130ms/step
4
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 131ms/step


'mi coche es nuevo'